# Pandas — Data Transformation

> **Repo:** Python_Libraries | **Library:** Pandas  
> **Notebook:** 05_Data_Transformation  

---

#### What is Data Transformation?

Data Transformation refers to changing the **structure, scale, or representation** of data to make it more suitable for analysis or machine learning.

Unlike Data Cleaning (which fixes errors), transformation **reshapes and enriches** clean data.

#### Topics Covered:

1. Applying Functions (`apply`, `map`, `applymap`)
2. Replacing Values
3. Binning & Discretization
4. Normalization & Scaling
5. Creating New Columns (Feature Engineering)
6. Renaming & Reindexing
7. Type Casting
8. Encoding Categorical Variables

In [1]:
import pandas as pd
import numpy as np

# Sample Employee DataFrame — used throughout this notebook
data = {
    'name':       ['Amit', 'Bhavna', 'Carlos', 'Diana', 'Elan'],
    'department': ['HR', 'IT', 'IT', 'Finance', 'HR'],
    'salary':     [42000, 85000, 92000, 61000, 47000],
    'experience': [2, 7, 9, 4, 3],
    'rating':     [3.2, 4.5, 4.8, 3.9, 3.5],
    'gender':     ['M', 'F', 'M', 'F', 'M']
}

df = pd.DataFrame(data)
df

,name,department,salary,experience,rating,gender
0,Amit,HR,42000,2,3.2,M
1,Bhavna,IT,85000,7,4.5,F
2,Carlos,IT,92000,9,4.8,M
3,Diana,Finance,61000,4,3.9,F
4,Elan,HR,47000,3,3.5,M



# Section# 1. Applying Functions

Pandas gives you three key tools to apply custom logic across your data:

| Method | Applied On | Use Case |
|---|---|---|
| `apply()` | Row or Column (Series or DataFrame) | Most flexible — custom functions |
| `map()` | Single Series only | Element-wise mapping/replacement |
| `applymap()` / `map()` (v2.1+) | Entire DataFrame element-wise | Apply same logic to every cell |

In [15]:
# apply() on a single column (Series)
# Use case: categorize salary into bands

def salary_band(s):
    if s < 50000:
        return 'Low'
    elif s < 80000:
        return 'Mid'
    else:
        return 'High'

df['salary_band'] = df['salary'].apply(salary_band)
df[['name', 'salary', 'salary_band']]

,name,salary,salary_band
0,Amit,42000,Low
1,Bhavna,85000,High
2,Carlos,92000,High
3,Diana,61000,Mid
4,Elan,47000,Low


In [17]:
# apply() with a lambda — concise one-liner version
# Use case: flag senior employees (experience > 5)

df['is_senior'] = df['experience'].apply(lambda x: True if x > 5 else False)
df[['name', 'experience', 'is_senior']]

,name,experience,is_senior
0,Amit,2,False
1,Bhavna,7,True
2,Carlos,9,True
3,Diana,4,False
4,Elan,3,False


In [19]:
# apply() across rows (axis=1)
# Use case: compute a bonus = salary * rating * 0.01

df['bonus'] = df.apply(lambda row: round(row['salary'] * row['rating'] * 0.01, 2), axis=1)
df[['name', 'salary', 'rating', 'bonus']]

,name,salary,rating,bonus
0,Amit,42000,3.2,1344.0
1,Bhavna,85000,4.5,3825.0
2,Carlos,92000,4.8,4416.0
3,Diana,61000,3.9,2379.0
4,Elan,47000,3.5,1645.0


In [21]:
# map() — element-wise on a Series
# Use case: expand department abbreviation or map codes to labels

gender_map = {'M': 'Male', 'F': 'Female'}
df['gender_full'] = df['gender'].map(gender_map)
df[['name', 'gender', 'gender_full']]

,name,gender,gender_full
0,Amit,M,Male
1,Bhavna,F,Female
2,Carlos,M,Male
3,Diana,F,Female
4,Elan,M,Male



# Section  2. Replacing Values

`replace()` is used to substitute specific values in a Series or DataFra
me.
It's more powerful than it looks — supports exact values, lists, and regex.

| Method | Use Case |
|---|---|
| `replace(old, new)` | Simple single value swap |
| `replace({dict})` | Multiple replacements at once |
| `replace(list, list)` | Positional list-to-list mapping |
| `replace(regex=True)` | Pattern-based replacement |

In [27]:
# Basic replace — single value
# Use case: standardize a department name

df['department'].replace('HR', 'Human Resources')

0    Human Resources
1                 IT
2                 IT
3            Finance
4    Human Resources
Name: department, dtype: object

In [29]:
# Dict-based replace — multiple at once (most common in real world)
df['department'] = df['department'].replace({
    'HR': 'Human Resources',
    'IT': 'Information Technology',
    'Finance': 'Finance & Accounts'
})

df[['name', 'department']]

,name,department
0,Amit,Human Resources
1,Bhavna,Information Technology
2,Carlos,Information Technology
3,Diana,Finance & Accounts
4,Elan,Human Resources


In [33]:
# List-to-list replace — positional mapping
# Use case: recode rating into grade labels

df['grade'] = df['rating'].replace(
    [3.2, 4.5, 4.8, 3.9, 3.5],
    ['C', 'A', 'A+', 'B', 'C+']
)

df[['name', 'rating', 'grade']]

,name,rating,grade
0,Amit,3.2,C
1,Bhavna,4.5,A
2,Carlos,4.8,A+
3,Diana,3.9,B
4,Elan,3.5,C+


In [35]:
# replace() with regex=True
# Use case: clean up string patterns (e.g. remove department suffixes)

df['dept_short'] = df['department'].replace(
    r' & Accounts| Technology| Resources',
    '',
    regex=True
)

df[['department', 'dept_short']]

,department,dept_short
0,Human Resources,Human
1,Information Technology,Information
2,Information Technology,Information
3,Finance & Accounts,Finance
4,Human Resources,Human


In [37]:
# replace() on entire DataFrame at once
# Use case: replace sentinel/placeholder values like -1 or 'unknown'

df_test = pd.DataFrame({
    'A': [1, -1, 3],
    'B': ['yes', 'unknown', 'yes'],
    'C': [-1, 2, -1]
})

df_test.replace(-1, np.nan).replace('unknown', np.nan)

,A,B,C
0,1.0,yes,NaN
1,NaN,NaN,2.0
2,3.0,yes,NaN


#### `replace()` vs `map()`

| | `map()` | `replace()` |
|---|---|---|
| Works on | Series only | Series + DataFrame |
| Unmapped values | Becomes `NaN` | Stays unchanged ✅ |
| Regex support | ❌ | ✅ |

## Section 3. Binning & Discretization

Binning converts **continuous numerical data into categorical buckets**.

This is heavily used in feature engineering for ML and EDA reporting.

| Function | How it bins | Use Case |
|---|---|---|
| `pd.cut()` | Equal-width intervals (you define edges) | Age groups, salary slabs |
| `pd.qcut()` | Equal-frequency intervals (equal number of values per bin) | Percentile-based ranking |

**pd.cut**
Use for logical numeric thresholds (e.g., age, temperature, income brackets)

**pd.qcut**
Use for ranking or percentiles (e.g., test scores into quartiles, customer spending tiers).

In [46]:
# pd.cut() — you define the bin boundaries
# Use case: group salary into fixed slabs

df['salary_slab'] = pd.cut(
    df['salary'],
    bins=[0, 50000, 75000, 100000],
    labels=['Low', 'Mid', 'High']
)

df[['name', 'salary', 'salary_slab']]

,name,salary,salary_slab
0,Amit,42000,Low
1,Bhavna,85000,High
2,Carlos,92000,High
3,Diana,61000,Mid
4,Elan,47000,Low


In [54]:
# pd.cut() with bins=3 — pandas auto-calculates equal-width ranges

df['salary_auto_bin'] = pd.cut(df['salary'], bins=3)
df[['name', 'salary', 'salary_auto_bin']]

,name,salary,salary_auto_bin
0,Amit,42000,"(41950.0, 58666.667]"
1,Bhavna,85000,"(75333.333, 92000.0]"
2,Carlos,92000,"(75333.333, 92000.0]"
3,Diana,61000,"(58666.667, 75333.333]"
4,Elan,47000,"(41950.0, 58666.667]"


In [56]:
# right=False → intervals open on the right [ )
# include_lowest=True → ensures the minimum value is included

df['salary_slab2'] = pd.cut(
    df['salary'],
    bins=[0, 50000, 75000, 100000],
    labels=['Low', 'Mid', 'High'],
    right=False,
    include_lowest=True
)

df[['name', 'salary', 'salary_slab2']]

,name,salary,salary_slab2
0,Amit,42000,Low
1,Bhavna,85000,High
2,Carlos,92000,High
3,Diana,61000,Mid
4,Elan,47000,Low


In [58]:
# pd.qcut() — splits data so each bin has equal number of values
# Use case: rank employees into performance quartiles

df['perf_quartile'] = pd.qcut(
    df['rating'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

df[['name', 'rating', 'perf_quartile']]

,name,rating,perf_quartile
0,Amit,3.2,Q1
1,Bhavna,4.5,Q3
2,Carlos,4.8,Q4
3,Diana,3.9,Q2
4,Elan,3.5,Q1


In [64]:
# pd.qcut() with custom quantile boundaries
# Use case: split into bottom 25%, middle 50%, top 25%

df['rating_tier'] = pd.qcut(
    df['rating'],
    q=[0, 0.25, 0.75, 1.0],
    labels=['Low', 'Mid', 'Top']
)

df[['name', 'rating', 'rating_tier']]

,name,rating,rating_tier
0,Amit,3.2,Low
1,Bhavna,4.5,Mid
2,Carlos,4.8,Top
3,Diana,3.9,Mid
4,Elan,3.5,Low


In [66]:
# Always good to inspect bin distribution after cutting
df['salary_slab'].value_counts().sort_index()

salary_slab
Low     2
Mid     1
High    2
Name: count, dtype: int64

#### `pd.cut()` vs `pd.qcut()`

| | `pd.cut()` | `pd.qcut()` |
|---|---|---|
| Bin width | Equal width | Equal frequency |
| You control | Bin edges | Number of quantiles |
| Risk | Unequal counts per bin | Unequal bin widths |
| Best for | Domain-defined ranges (age, salary) | Percentile ranking, fair stribution.

**Common mistake:**
Using `pd.cut()` on skewed data → most values fall in one bin.

`pd.qcut()` guarantees balanced bins regardless of distribution.

## Section 4. Normalization & Scaling

When features have very different ranges (e.g., salary in lakhs vs. rating out of 5), ML models can get biased toward larger numbers.

**Scaling** brings all features to a **comparable range**.

| Technique | Formula | Output Range | Best For |
|---|---|---|---|
| Min-Max Normalization | (x - min) / (max - min) | 0 to 1 | Bounded, no outliers |
| Z-Score Standardization | (x - mean) / std | Unbounded (mean=0) | Normally distributed data |
| Robust Scaling | (x - median) / IQR | Unbounded | Data with outliers |

### 4.1 Min-Max Normalization

- scales values between 0 and 1
- Formula: (x - min) / (max - min)

In [73]:
df['salary_minmax'] = (df['salary'] - df['salary'].min()) / \
                      (df['salary'].max() - df['salary'].min())

df['experience_minmax'] = (df['experience'] - df['experience'].min()) / \
                          (df['experience'].max() - df['experience'].min())

df[['name', 'salary', 'salary_minmax', 'experience', 'experience_minmax']].round(4)

,name,salary,salary_minmax,experience,experience_minmax
0,Amit,42000,0.00,2,0.0000
1,Bhavna,85000,0.86,7,0.7143
2,Carlos,92000,1.00,9,1.0000
3,Diana,61000,0.38,4,0.2857
4,Elan,47000,0.10,3,0.1429


### 4.2 Z-score Standardization

- Centres data around mean = 0 and std = 1
- Formula: ( x - mean ) / std

In [76]:
df['salary_zscore'] = (df['salary'] - df['salary'].mean()) / df['salary'].std()
df['rating_zscore'] = (df['rating'] - df['rating'].mean()) / df['rating'].std()

df[['name', 'salary', 'salary_zscore', 'rating', 'rating_zscore']].round(4)

,name,salary,salary_zscore,rating,rating_zscore
0,Amit,42000,-1.0472,3.2,-1.1667
1,Bhavna,85000,0.8772,4.5,0.7778
2,Carlos,92000,1.1904,4.8,1.2265
3,Diana,61000,-0.1969,3.9,-0.1197
4,Elan,47000,-0.8234,3.5,-0.7179


### 4.3 Robust Scaling:

- Using Median and IQR, resistant to outliers
- Formula: ( x - median ) / IQR

In [79]:
Q1 = df['salary'].quantile(0.25)
Q3 = df['salary'].quantile(0.75)
IQR = Q3 - Q1

df['salary_robust'] = (df['salary'] - df['salary'].median()) / IQR

df[['name', 'salary', 'salary_robust']].round(4)

,name,salary,salary_robust
0,Amit,42000,-0.5000
1,Bhavna,85000,0.6316
2,Carlos,92000,0.8158
3,Diana,61000,0.0000
4,Elan,47000,-0.3684


### 4.4 Scaling Multiple Numeric columns at once

In [82]:
# Using a reusable function
def minmax_scale(series):
    return (series - series.min()) / (series.max() - series.min())

cols_to_scale = ['salary', 'experience', 'rating']
df_scaled = df[cols_to_scale].apply(minmax_scale).round(4)
df_scaled.columns = [c + '_scaled' for c in cols_to_scale]

pd.concat([df['name'], df_scaled], axis=1)

,name,salary_scaled,experience_scaled,rating_scaled
0,Amit,0.00,0.0000,0.0000
1,Bhavna,0.86,0.7143,0.8125
2,Carlos,1.00,1.0000,1.0000
3,Diana,0.38,0.2857,0.4375
4,Elan,0.10,0.1429,0.1875


#### Which scaling to use?

| Situation | Use |
|---|---|
| No outliers, need 0–1 range | Min-Max |
| Data is roughly normal | Z-Score |
| Outliers present | Robust Scaling |
| Tree-based models (Random Forest, XGBoost) | No scaling needed |
| Linear models, KNN, SVM, Neural Nets | Always scale |

**Key point:** Scaling doesn't change the *shape* of the distribution —
it only changes the *range*. Skewed data stays skewed after scaling.

## Section 5. Creating New Columns (Feature Engineering)

Feature Engineering means **deriving new meaningful columns** from existing ones.

It's one of the most impactful steps in any real-world data pipeline.

We cover three core tools here:

| Tool | Use Case |
|---|---|
| Direct arithmetic | Simple derived columns |
| `np.where()` | Single condition → two outcomes |
| `np.select()` | Multiple conditions → multiple outcomes |
| `pd.Series.str` methods | String-based derived features |

### 5.1 Direct Arithmetic

simplest form of Feature Engineering

In [89]:
df['salary_monthly'] = (df['salary'] / 12).round(2)   # Creating Monthly Salary column by dividing salary / 12 months
df['experience_bonus'] = df['experience'] * 1000   #  Giving Experience Bonus as $1000 per year of experience
df['total_comp'] = df['salary'] + df['bonus'] + df['experience_bonus'] # Combining all compensations together -- Salary, Bonus and Experience bonus

df[['name', 'salary', 'salary_monthly', 'bonus', 'experience_bonus', 'total_comp']]

,name,salary,salary_monthly,bonus,experience_bonus,total_comp
0,Amit,42000,3500.00,1344.0,2000,45344.0
1,Bhavna,85000,7083.33,3825.0,7000,95825.0
2,Carlos,92000,7666.67,4416.0,9000,105416.0
3,Diana,61000,5083.33,2379.0,4000,67379.0
4,Elan,47000,3916.67,1645.0,3000,51645.0


### 5.2 Conditional FE using `np.where`:

- Syntax: np.where(condition, value_if_true, value_if_false)

In [93]:
df['high_earner'] = np.where(df['salary'] >= 75000, 'Yes', 'No')
df[['name', 'salary', 'high_earner']]

,name,salary,high_earner
0,Amit,42000,No
1,Bhavna,85000,Yes
2,Carlos,92000,Yes
3,Diana,61000,No
4,Elan,47000,No


In [75]:
# Nested np.where() — handles 3 outcomes
# Use case: experience level tag

df['exp_level'] = np.where(
    df['experience'] <= 3, 'Junior',
    np.where(df['experience'] <= 6, 'Mid', 'Senior')
)

df[['name', 'experience', 'exp_level']]

,name,experience,exp_level
0,Amit,2,Junior
1,Bhavna,7,Senior
2,Carlos,9,Senior
3,Diana,4,Mid
4,Elan,3,Junior


### 5.3 Multiple Conditions: using `np.select()`

In [102]:
# cleaner than nested np.where() for 3+ conditions
# Use case: performance label based on rating

conditions = [
    df['rating'] >= 4.5,
    df['rating'] >= 3.8,
    df['rating'] >= 3.0
]

choices = ['Excellent', 'Good', 'Average']

df['performance'] = np.select(conditions, choices, default='Poor')
df[['name', 'rating', 'performance']]

,name,rating,performance
0,Amit,3.2,Average
1,Bhavna,4.5,Excellent
2,Carlos,4.8,Excellent
3,Diana,3.9,Good
4,Elan,3.5,Average


### 5.4 String Based derived Features: using `.str`

In [106]:
# Deriving features from string columns
# Use case: extract first name initial, name length

df['name_initial'] = df['name'].str[0]
df['name_length']  = df['name'].str.len()

df[['name', 'name_initial', 'name_length']]

,name,name_initial,name_length
0,Amit,A,4
1,Bhavna,B,6
2,Carlos,C,6
3,Diana,D,5
4,Elan,E,4


### 5.5 Combining Multiple Columns 

In [109]:
# Combining multiple columns into one derived flag
# Use case: identify "high value" employees

df['high_value'] = np.where(
    (df['salary'] >= 60000) & (df['rating'] >= 4.0),
    'High Value',
    'Standard'
)

df[['name', 'salary', 'rating', 'high_value']]

,name,salary,rating,high_value
0,Amit,42000,3.2,Standard
1,Bhavna,85000,4.5,High Value
2,Carlos,92000,4.8,High Value
3,Diana,61000,3.9,Standard
4,Elan,47000,3.5,Standard


### `np.where()` vs `np.select()`

| | `np.where()` | `np.select()` |
|---|---|---|
| Conditions | Single (nestable) | Multiple, clean list |
| Outcomes | 2 (true/false) | Many (one per condition) |
| Readability | Gets messy when nested | Clean for 3+ conditions |
| Default value | The false value | Separate `default=` t()` instead

**Rule of thumb:**

- 1 condition → `np.where()`
- 2+ conditions → `np.select()`
- Never nest `np.where()` more than once — use `np.select()` instead

## Section 6. Renaming & Reindexing

Renaming and reindexing help you **standardize column/index labels** and **restructure the shape** of your DataFrame for downstream use.

| Method | Purpose |
|---|---|
| `rename()` | Rename specific columns or index labels |
| `rename(str.lower)` | Bulk rename using a function |
| `set_index()` | Promote a column to become the index |
| `reset_index()` | Flatten index back into a column |
| `reindex()` | Reorder or add rows/columns by label |

### 6.1 `.rename()` : Renaming specific columns

In [121]:
# rename() — rename specific columns by dict mapping
df_r = df[['name', 'salary', 'rating', 'department']].copy()

df_r = df_r.rename(columns={
    'name':       'employee_name',
    'salary':     'annual_salary',
    'rating':     'performance_rating',
    'department': 'dept'
})

df_r

,employee_name,annual_salary,performance_rating,dept
0,Amit,42000,3.2,Human Resources
1,Bhavna,85000,4.5,Information Technology
2,Carlos,92000,4.8,Information Technology
3,Diana,61000,3.9,Finance & Accounts
4,Elan,47000,3.5,Human Resources


### 6.2 Renaming columns with a function using `.rename()`

In [126]:
# Use case: lowercase all columns, strip spaces (very common in real pipelines)

df_r.columns
df_r = df_r.rename(columns=str.title)   # or str.lower, str.strip
df_r

,Employee_Name,Annual_Salary,Performance_Rating,Dept
0,Amit,42000,3.2,Human Resources
1,Bhavna,85000,4.5,Information Technology
2,Carlos,92000,4.8,Information Technology
3,Diana,61000,3.9,Finance & Accounts
4,Elan,47000,3.5,Human Resources


### 6.3 `set_index()`:

Setting an existing column as the index of the DataFrame

In [131]:
# set_index() — make 'name' the row index instead of 0,1,2...
df_indexed = df[['name', 'salary', 'rating']].set_index('name')
df_indexed

,salary,rating
name,,
Amit,42000,3.2
Bhavna,85000,4.5
Carlos,92000,4.8
Diana,61000,3.9
Elan,47000,3.5


### 6.4 `reset_index()`:

bringing the index back as a regular column

In [134]:
df_indexed.reset_index()

,name,salary,rating
0,Amit,42000,3.2
1,Bhavna,85000,4.5
2,Carlos,92000,4.8
3,Diana,61000,3.9
4,Elan,47000,3.5


## 6.5 `.reindex()`:

> reorder columns explicitly in a desired order

In [137]:
# Use case: enforce a standard column order before exporting

desired_order = ['name', 'department', 'experience', 'salary', 'rating', 'performance']
df_reordered = df[desired_order].reindex(columns=desired_order)
df_reordered

,name,department,experience,salary,rating,performance
0,Amit,Human Resources,2,42000,3.2,Average
1,Bhavna,Information Technology,7,85000,4.5,Excellent
2,Carlos,Information Technology,9,92000,4.8,Excellent
3,Diana,Finance & Accounts,4,61000,3.9,Good
4,Elan,Human Resources,3,47000,3.5,Average


In [139]:
# reindex() on rows — fills in missing index labels with NaN
# Use case: align two DataFrames that may have different index ranges

df_small = df[['name', 'salary']].copy()
df_small.index = [10, 20, 30, 40, 50]   # non-standard index

df_small.reindex([10, 20, 30, 40, 50, 60, 70])  # 60 and 70 don't exist → NaN

,name,salary
10,Amit,42000.0
20,Bhavna,85000.0
30,Carlos,92000.0
40,Diana,61000.0
50,Elan,47000.0
60,NaN,NaN
70,NaN,NaN


### `set_index()` vs `reset_index()`

| | `set_index()` | `reset_index()` |
|---|---|---|
| What it does | Moves a column → index | Moves index → column |
| When to use | Faster label-based lookup | After groupby, merge, or filter |
| `inplace=` | Available | Available |
| `drop=True` | N/A | Drops old index instead of adding it as can flat table.

**Common real-world pattern:**
```python
df.groupby('department')['salary'].mean().reset_index()
```
`groupby()` creates a named index — `reset_index()` brings it back as a column so the result looks like a clean flat table.

## Section 7: Type Casting

Pandas often infers dtypes incorrectly on import.

Type casting lets you **explicitly control how data is stored**, 

which affects memory usage, computation speed, and correctness.

| Method | Purpose |
|---|---|
| `astype()` | Convert column to a specific dtype |
| `pd.to_numeric()` | Safely convert to number, handle errors |
| `pd.to_datetime()` | Convert strings to datetime |
| `pd.to_timedelta()` | Convert to duration type |
| `.dtypes` | Inspect current dtypes |
| `select_dtypes()` | Filter columns by dtype |

### 7.1 Inspecting columns using `.dtypes` 

In [8]:
# Always start by inspecting current dtypes
df[['name', 'salary', 'experience', 'rating']].dtypes

name           object
salary          int64
experience      int64
rating        float64
dtype: object

### 7.2 Explicit Coversion using `.astype()`

In [11]:
# astype() — explicit conversion
df['salary']     = df['salary'].astype('float64')
df['experience'] = df['experience'].astype('int32')   # saves memory vs int64

df[['salary', 'experience']].dtypes

salary        float64
experience      int32
dtype: object

In [13]:
# Converting string columns to 'category' dtype
# Use case: saves significant memory for low-cardinality columns

df['department'] = df['department'].astype('category')
df['gender']     = df['gender'].astype('category')

df[['department', 'gender']].dtypes

department    category
gender        category
dtype: object

### 7.3 `pd.to_numeric()`: Safer Conversion of Numeric Columns

In [23]:
# pd.to_numeric() — safer than astype() when data may have errors
# errors='coerce' turns invalid values into NaN instead of raising

s = pd.Series(['42000', '85000', 'N/A', '61000', 'unknown'])

pd.to_numeric(s, errors='coerce')   # 'N/A' and 'unknown' → NaN

0    42000.0
1    85000.0
2        NaN
3    61000.0
4        NaN
dtype: float64

### 7.4 `pd.to_datetime()`: Coversion of Date Columns

In [34]:
# pd.to_datetime() — convert string dates to datetime objects

df['join_date'] = ['2022-01-15', '2018-06-01', '2016-03-22',
                   '2021-09-10', '2022-11-05']

df['join_date'] = pd.to_datetime(df['join_date'])

# Now you can extract components -- otherwise using `.dt` will throw an error 
df['join_year']  = df['join_date'].dt.year
df['join_month'] = df['join_date'].dt.month

df[['name', 'join_date', 'join_year', 'join_month']]

,name,join_date,join_year,join_month
0,Amit,2022-01-15,2022,1
1,Bhavna,2018-06-01,2018,6
2,Carlos,2016-03-22,2016,3
3,Diana,2021-09-10,2021,9
4,Elan,2022-11-05,2022,11


### 7.5 Selecting specific columns by their data type: `select_dtypes`

In [47]:
# select_dtypes() — filter columns by their dtype
# Very useful when you want to apply operations only to numeric columns

df.select_dtypes(include='number')      # all numeric columns 

,salary,experience,rating,join_year,join_month
0,42000.0,2,3.2,2022,1
1,85000.0,7,4.5,2018,6
2,92000.0,9,4.8,2016,3
3,61000.0,4,3.9,2021,9
4,47000.0,3,3.5,2022,11


In [49]:
df.select_dtypes(exclude='number')       # everything except numeric

,name,department,gender,join_date
0,Amit,HR,M,2022-01-15
1,Bhavna,IT,F,2018-06-01
2,Carlos,IT,M,2016-03-22
3,Diana,Finance,F,2021-09-10
4,Elan,HR,M,2022-11-05


In [51]:
df.select_dtypes(include='object')       # all string columns

,name
0,Amit
1,Bhavna
2,Carlos
3,Diana
4,Elan


### `astype()` vs `pd.to_numeric()`

| | `astype()` | `pd.to_numeric()` |
|---|---|---|
| On invalid value | Raises error ❌ | Can coerce to NaN ✅ |
| Use when | Data is clean and known | Data may have dirty strings |
| Category dtype | ✅ supported | ❌ not applicue values.

**Memory tip:** 

- Use `int32` / `float32` instead of default `int64` / `float64`
- for large datasets — cuts memory usage by 50%.
- Use `category` dtype for string columns with few unique values.

## Section 8: Encoding Categorical Variables 

ML models cannot work with raw text — categories must be converted to numbers.

This process is called **Encoding**.

| Method | Output | Best For |
|---|---|---|
| `pd.get_dummies()` | Binary 0/1 columns per category | Nominal data (no order) |
| Manual label encoding via `map()` | Single integer column | Ordinal data (has order) |
| `pd.factorize()` | Integer codes + index array | Quick encoding, EDA |
| `.cat.codes` | Integer codes from category dtype | When dtype is already 'category' |

### 8.1 `pd.get_dummies()`: One-Hot Encoding

In [58]:
# pd.get_dummies() — One-Hot Encoding
# Creates a binary column for each unique category

df_encoded = pd.get_dummies(df[['name', 'gender', 'department']], 
                             columns=['gender', 'department'])
df_encoded

,name,gender_F,gender_M,department_Finance,department_HR,department_IT
0,Amit,False,True,False,True,False
1,Bhavna,True,False,False,False,True
2,Carlos,False,True,False,False,True
3,Diana,True,False,True,False,False
4,Elan,False,True,False,True,False


In [69]:
# drop_first=True — drops the first dummy to avoid multicollinearity
# This is important for linear models (regression, logistic regression)

df_encoded2 = pd.get_dummies(df[['name', 'gender']], 
                              columns=['gender'], 
                              drop_first=True)
df_encoded2

,name,gender_M
0,Amit,True
1,Bhavna,False
2,Carlos,True
3,Diana,False
4,Elan,True


### 8.2 Manual Label Encoding using `.map`

In [79]:
# Manual label encoding — for ORDINAL data (order matters)
# Use case: encode exp_level which has a natural order

ordinal_map = {'Junior': 1, 'Mid': 2, 'Senior': 3}
df['exp_level_encoded'] = df['exp_level'].map(ordinal_map)

df[['name', 'exp_level', 'exp_level_encoded']]

,name,exp_level,exp_level_encoded
0,Amit,Junior,1
1,Bhavna,Senior,3
2,Carlos,Senior,3
3,Diana,Mid,2
4,Elan,Junior,1


### 8.3 `pd.factorize()`: Quick Encoding

In [3]:
# pd.factorize() — quick encoding, returns codes + unique labels
# Use case: fast EDA encoding without creating a full mapping dict

codes, uniques = pd.factorize(df['department'])

print("Codes  :", codes)
print("Uniques:", uniques)

df['dept_code'] = codes
df[['name', 'department', 'dept_code']]

Codes  : [0 1 1 2 0]
Uniques: Index(['HR', 'IT', 'Finance'], dtype='object')


,name,department,dept_code
0,Amit,HR,0
1,Bhavna,IT,1
2,Carlos,IT,1
3,Diana,Finance,2
4,Elan,HR,0


#### Note:

- `pd.factorize()` is used to encode categorical variables into integer codes by assigning a unique number to each distinct value based on order of appearance.
- It returns two outputs: an `array of codes` and the `corresponding unique values`.
- It is mainly used for quick encoding during EDA
- but not recommended directly for machine learning models due to lack of ordinal meaning.

### 8.4 `.cat.codes`: Memory Efficient Encoding on 'Categorical' columns

In [33]:
# .cat.codes — works when column dtype is already 'category'
# Use case: memory-efficient encoding on large categorical columns
df['department'] = df['department'].astype('category')

df['dept_cat_code'] = df['department'].cat.codes
df[['name', 'department', 'dept_cat_code']]

,name,department,dept_cat_code
0,Amit,HR,1
1,Bhavna,IT,2
2,Carlos,IT,2
3,Diana,Finance,0
4,Elan,HR,1


### Encoding Methods Compared

| Method | Nominal | Ordinal | Interpretable | Multicollinearity Risk |
|---|---|---|---|---|
| `get_dummies()` | ✅ | ❌ | ✅ | Yes — use drop_first=True |
| `map()` (manual) | ❌ | ✅ | ✅ | No |
| `factorize()` | ✅ | ❌ | ⚠️ (arbitrary codes) | No |
| `.cat.codes` | ✅ | ❌ | ⚠️ (arbitrary codes) | No |

**Key rule:**
- Nominal (no order) → `get_dummies()`
- Ordinal (has order) → manual `map()` with meaningful integers
- Never use `factorize()` or `.cat.codes` for ML models directly, the codes are arbitrary and may mislead the model
- Nominal data consists of categories without any inherent order, like departments or colors.
- Multicollinearity occurs when features are highly correlated or redundant, causing instability in model coefficients.
- Nominal = Names only (no order)
- Multicollinearity = Duplicate information


##  Notebook Summary — Data Transformation with Pandas

| # | Topic | Key Methods |
|---|---|---|
| 1 | Applying Functions | `apply()`, `map()`, lambda |
| 2 | Replacing Values | `replace()`, dict/list/regex |
| 3 | Binning | `pd.cut()`, `pd.qcut()` |
| 4 | Scaling | Min-Max, Z-Score, Robust |
| 5 | Feature Engineering | `np.where()`, `np.select()`, arithmetic |
| 6 | Renaming & Reindexing | `rename()`, `set_index()`, `reindex()` |
| 7 | Type Casting | `astype()`, `to_numeric()`, `to_datetime()` |
| 8 | Encoding | `get_dummies()`, `map()`, `faregation.ipynb